# Fine-tune BERT for Token Classification (NER) — HF Trainer

This is the "easy mode" approach: Hugging Face's `Trainer` handles the training loop,
optimizer, scheduler, checkpointing, and logging for you.

Compare against `bert_crf_raw.ipynb`, which does the same task with a hand-written
PyTorch loop and a CRF output layer instead of plain softmax.


In [ ]:
!pip install -q transformers datasets seqeval evaluate accelerate

## 1. Config

In [ ]:
MODEL_NAME = "bert-base-cased"
# Note: the original "conll2003" repo uses a legacy loading script that
# datasets>=4.0 no longer supports. Using a script-free mirror instead.
DATASET_NAME = "tomaarsen/conll2003"   # swap for your own dataset
OUTPUT_DIR = "./bert-ner-trainer"

## 2. Load data

Using CoNLL-2003 as a stand-in. If you have your own data, it needs a `tokens` column
(list of words per example) and an `ner_tags` column (list of integer label ids per word).

In [ ]:
from datasets import load_dataset

raw_datasets = load_dataset(DATASET_NAME)
label_names = raw_datasets["train"].features["ner_tags"].feature.names
id2label = {i: l for i, l in enumerate(label_names)}
label2id = {l: i for i, l in enumerate(label_names)}
num_labels = len(label_names)

print(label_names)
raw_datasets["train"][0]

## 3. Tokenize and align labels

BERT's tokenizer splits words into subwords. We need to map each original NER tag onto the
right subword token, and mask the rest with `-100` so the loss ignores them.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # special tokens ([CLS], [SEP], padding)
            elif word_id != prev_word_id:
                label_ids.append(labels[word_id])  # first subword of a word gets the real tag
            else:
                label_ids.append(-100)  # subsequent subwords of the same word are ignored
            prev_word_id = word_id
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
tokenized_datasets

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

## 4. Model

Plain linear classification head per token, **no CRF**. Each token's tag is predicted
independently (softmax over labels), with no explicit modeling of tag-to-tag transitions.

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

## 5. Metrics

`seqeval` gives entity-level precision/recall/F1, not just per-token accuracy — that's
what you actually care about for NER.

In [ ]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    true_predictions = [
        [id2label[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 6. Train

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

## 7. Evaluate on test set + save

In [ ]:
print(trainer.evaluate(tokenized_datasets["test"]))
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## Contrast: HF Trainer vs. raw-PyTorch BERT+CRF

**1. Output layer**
- *Trainer:* a single `nn.Linear(hidden_size, num_labels)` on top of BERT. Each token's
  label is predicted independently via softmax.
- *CRF:* same linear layer produces "emission scores," but a CRF layer adds a learned
  transition-score matrix (`num_labels x num_labels`) and computes the *globally* best
  label sequence via the Viterbi algorithm, instead of picking each token's argmax in
  isolation — letting it structurally learn things like "an I-PER tag can't follow an O tag."

**2. Loss function**
- *Trainer:* standard token-level cross-entropy, computed automatically inside
  `AutoModelForTokenClassification.forward()` whenever you pass `labels=...`.
- *CRF:* negative log-likelihood of the *whole tag sequence* under the CRF (via the
  forward algorithm) — a different quantity than summed per-token cross-entropy. You
  implement/import this (e.g. `pytorch-crf`) and wire it into a custom forward pass.

**3. Training loop**
- *Trainer:* `Trainer.train()` handles the loop, gradient accumulation, mixed precision,
  checkpointing, logging, multi-GPU/distributed training, and LR scheduling via
  `TrainingArguments`.
- *CRF:* hand-rolled loop — you manually handle device placement, gradient clipping,
  scheduler stepping, checkpoint saving, and eval loops.

**4. Decoding at inference time**
- *Trainer:* argmax over logits per token. Cheap (`O(seq_len)`), but can produce
  invalid/inconsistent label sequences.
- *CRF:* Viterbi decoding for the single most probable *valid* tag sequence
  (`O(seq_len * num_labels^2)`), guaranteeing structurally consistent output.

**5. Code volume & flexibility tradeoff**
- *Trainer:* most of this notebook is data prep, not training logic. Very little control
  over the inner loop.
- *CRF:* usually 2-3x more code once you add the CRF layer, custom Dataset/DataLoader,
  manual padding/masking, and your own train/eval loops — but full control (constrained
  decoding, custom transition penalties, swapping in BiLSTM-CRF, etc).

**6. When CRF actually wins**
- Small/noisy datasets where label-transition structure (BIO scheme validity) matters a
  lot and there isn't enough data for BERT to learn it implicitly.
- Tasks with strict structural constraints (nested or schema-constrained tagging).
- With enough fine-tuning data, plain BERT (no CRF) often gets very close to BERT+CRF on
  F1 — BERT's contextual embeddings already encode much of the local structure a CRF
  would otherwise need to learn. CRF's edge shows up most when data is scarce.
